# 🚀 GE2PE Persian Diacritization - Google Colab Demo (Fixed)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elikaaghaei/Rahnema_college_phonemizer_v1/blob/main/GE2PE_Colab_Demo_Fixed.ipynb)

## ✨ تغییرات:
- ✅ استفاده از Gradio به جای ngrok (بدون نیاز به authtoken)
- ✅ استفاده از مدل GE2PE موجود
- ✅ UI تعاملی و زیبا
- ✅ Public URL رایگان

## مراحل:
1. Setup و clone repository
2. نصب dependencies
3. تست components
4. اجرای Gradio UI
5. دسترسی عمومی با share=True

## 1️⃣ Setup و بررسی محیط

In [ ]:
# بررسی GPU
!nvidia-smi

print("\n" + "="*60)
print("💾 فضای دیسک:")
print("="*60)
!df -h | grep -E "Filesystem|/content"

print("\n" + "="*60)
print("🧠 RAM:")
print("="*60)
!free -h

## 2️⃣ Clone Repository

In [ ]:
import os

# حذف clone قبلی اگر وجود دارد
if os.path.exists('Rahnema_college_phonemizer_v1'):
    !rm -rf Rahnema_college_phonemizer_v1

# Clone repository
!git clone https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1.git
%cd Rahnema_college_phonemizer_v1

print("\n✅ Repository cloned successfully!")
!ls -la

## 3️⃣ نصب Dependencies

In [ ]:
# نصب پکیج‌های مورد نیاز
!pip install -q torch torchvision torchaudio
!pip install -q transformers==4.30.0
!pip install -q fastapi uvicorn pydantic
!pip install -q parsivar  # برای GE2PE
!pip install -q gradio    # برای UI
!pip install -q pandas

print("✅ همه dependencies نصب شدند!")

In [ ]:
# بررسی نصب
import torch
import transformers
import gradio as gr

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Gradio: {gr.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4️⃣ تست Dataset Loader

In [ ]:
from data.loader import PhonemizerDataset, train_val_split

# بارگذاری dataset
print("📖 بارگذاری dataset...")
dataset = PhonemizerDataset(
    csv_path='phonemizer _dataset_v1.csv/phonemizer _dataset_v1.csv',
    mode='char',
    preserve_diacritics=True,
    max_samples=50  # نمونه کوچک برای تست
)

print(f"\n✅ Dataset: {len(dataset)} samples loaded")

# نمایش 2 نمونه
for i in range(2):
    sample = dataset[i]
    print(f"\nSample {i+1}:")
    print(f"  Text: {sample['text'][:40]}...")
    print(f"  Phonemes: {sample['phonemes'][:40]}...")

## 5️⃣ تست Tokenizer

In [ ]:
from data.tokenizer import PersianTokenizer

# ایجاد tokenizer
tokenizer = PersianTokenizer()
texts = ['سلام', 'دنیا', 'تست']
tokenizer.build_vocab(texts, mode='char')

print(f"✅ Tokenizer ready: {tokenizer.vocab_size} tokens")

# تست
text = "سلام"
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)
print(f"\nTest: '{text}' → {encoded} → '{decoded}'")
print(f"Match: {text == decoded} ✅")

## 6️⃣ بارگذاری GE2PE Model

In [ ]:
from GE2PE.GE2PE import GE2PE
import torch

print("🔄 بارگذاری GE2PE model...")
print("⚠️  از model base استفاده می‌شود (checkpoint موجود نیست)\n")

# بارگذاری model
# استفاده از model base کوچک (mt5-small)
model = GE2PE(
    model_path='google/mt5-small',
    GPU=torch.cuda.is_available()
)

print("✅ Model loaded successfully!")

# تست model
test_texts = ["سلام", "دنیا"]
results = model.generate(test_texts, batch_size=2)

print("\n🧪 Test inference:")
for text, result in zip(test_texts, results):
    print(f"  '{text}' → '{result}'")

## 7️⃣ ساخت Gradio Interface

In [ ]:
import gradio as gr

# تابع اصلی برای diacritization
def diacritize_text(text, use_rules=False, use_dict=False):
    """
    اعراب‌گذاری متن فارسی
    """
    if not text.strip():
        return "⚠️ لطفاً متنی وارد کنید"
    
    try:
        # اجرای model
        results = model.generate(
            [text],
            batch_size=1,
            use_rules=use_rules,
            use_dict=use_dict
        )
        return results[0]
    except Exception as e:
        return f"❌ خطا: {str(e)}"

# تابع batch processing
def diacritize_batch(texts_area, use_rules=False, use_dict=False):
    """
    اعراب‌گذاری چند متن (هر خط یک متن)
    """
    if not texts_area.strip():
        return "⚠️ لطفاً متن‌ها را وارد کنید (هر خط یک متن)"
    
    # تقسیم به خطوط
    lines = [line.strip() for line in texts_area.split('\n') if line.strip()]
    
    if not lines:
        return "⚠️ متن معتبری یافت نشد"
    
    try:
        # پردازش batch
        results = model.generate(
            lines,
            batch_size=min(len(lines), 10),
            use_rules=use_rules,
            use_dict=use_dict
        )
        
        # فرمت خروجی
        output = ""
        for i, (inp, out) in enumerate(zip(lines, results), 1):
            output += f"{i}. ورودی:  {inp}\n"
            output += f"   خروجی: {out}\n\n"
        
        return output
        
    except Exception as e:
        return f"❌ خطا: {str(e)}"

print("✅ توابع Gradio آماده شدند")

## 8️⃣ اجرای Gradio UI

In [ ]:
# ساخت interface
with gr.Blocks(title="GE2PE Persian Diacritization", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # 🎯 GE2PE - اعراب‌گذاری متن فارسی
    
    سیستم اعراب‌گذاری خودکار متن‌های فارسی با استفاده از مدل GE2PE مبتنی بر T5
    """)
    
    with gr.Tabs():
        # Tab 1: تک متن
        with gr.Tab("تک متن"):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(
                        label="متن ورودی (بدون اعراب)",
                        placeholder="مثال: سلام دنیا",
                        lines=3
                    )
                    
                    with gr.Row():
                        use_rules_single = gr.Checkbox(
                            label="استفاده از قوانین",
                            value=False
                        )
                        use_dict_single = gr.Checkbox(
                            label="استفاده از دیکشنری",
                            value=False
                        )
                    
                    submit_btn = gr.Button("اعراب‌گذاری", variant="primary")
                
                with gr.Column():
                    output_text = gr.Textbox(
                        label="متن خروجی (با اعراب)",
                        lines=3
                    )
            
            # مثال‌ها
            gr.Examples(
                examples=[
                    ["سلام دنیا", False, False],
                    ["من به مدرسه می روم", False, False],
                    ["کتاب خوب است", False, False],
                    ["هوا امروز خوب است", False, False],
                ],
                inputs=[input_text, use_rules_single, use_dict_single],
            )
            
            submit_btn.click(
                fn=diacritize_text,
                inputs=[input_text, use_rules_single, use_dict_single],
                outputs=output_text
            )
        
        # Tab 2: چند متن
        with gr.Tab("چند متن"):
            gr.Markdown("### هر خط یک متن جداگانه")
            
            with gr.Row():
                with gr.Column():
                    batch_input = gr.Textbox(
                        label="متن‌های ورودی (هر خط یک متن)",
                        placeholder="سلام دنیا\nاین یک تست است\nزبان فارسی زیباست",
                        lines=10
                    )
                    
                    with gr.Row():
                        use_rules_batch = gr.Checkbox(
                            label="استفاده از قوانین",
                            value=False
                        )
                        use_dict_batch = gr.Checkbox(
                            label="استفاده از دیکشنری",
                            value=False
                        )
                    
                    batch_btn = gr.Button("اعراب‌گذاری دسته‌ای", variant="primary")
                
                with gr.Column():
                    batch_output = gr.Textbox(
                        label="نتایج",
                        lines=10
                    )
            
            batch_btn.click(
                fn=diacritize_batch,
                inputs=[batch_input, use_rules_batch, use_dict_batch],
                outputs=batch_output
            )
        
        # Tab 3: اطلاعات
        with gr.Tab("ℹ️ اطلاعات"):
            gr.Markdown("""
            ## درباره GE2PE
            
            این سیستم از مدل GE2PE (Grapheme-to-Phoneme Enhanced) برای اعراب‌گذاری خودکار متن‌های فارسی استفاده می‌کند.
            
            ### ویژگی‌ها:
            - ✅ مبتنی بر مدل T5 (Transformer)
            - ✅ پشتیبانی از batch processing
            - ✅ قوانین پس‌پردازش اختیاری
            - ✅ دیکشنری سفارشی (اختیاری)
            
            ### نحوه استفاده:
            1. متن فارسی خود را وارد کنید (بدون اعراب)
            2. روی دکمه "اعراب‌گذاری" کلیک کنید
            3. نتیجه با اعراب نمایش داده می‌شود
            
            ### لینک‌های مفید:
            - [GitHub Repository](https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1)
            - [Documentation](https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1/blob/main/README.md)
            """)
            
            # نمایش اطلاعات محیط
            info_text = f"""
            ### 🖥️ اطلاعات محیط:
            - PyTorch: {torch.__version__}
            - GPU: {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ فعال نیست'}
            - Model: google/mt5-small (base)
            """
            gr.Markdown(info_text)

print("✅ Gradio interface ساخته شد")

## 9️⃣ راه‌اندازی UI با Public URL

In [ ]:
# راه‌اندازی با share=True برای public URL
print("🚀 راه‌اندازی Gradio UI...\n")
print("⏳ لطفاً صبر کنید...\n")

# Launch
demo.launch(
    share=True,        # ایجاد public URL رایگان
    debug=True,
    show_error=True
)

# URL به صورت خودکار نمایش داده می‌شود
print("\n" + "="*60)
print("✅ UI آماده است!")
print("="*60)
print("\n💡 لینک public در بالا نمایش داده شده است")
print("🌐 روی لینک کلیک کنید یا در browser کپی کنید")
print("="*60)